# 04. 聚类与降维体系

无监督学习不依赖标签，更强调数据内部结构的发现。

## 先建立直觉

            无监督学习最像“看一堆没有标注的东西，试着自己找规律”。

聚类在问的是：

- 哪些样本彼此更像？
- 数据里是不是天然存在几类群体？

降维在问的是：

- 这些高维特征里，真正有用的信息是不是只集中在少数方向上？

所以无监督学习的核心，不是预测标签，而是理解数据结构本身。

## 架构是怎么工作的

            这一节会出现三种完全不同的思路：

- KMeans：假设数据围绕若干中心聚集
- DBSCAN：假设高密度区域形成簇
- PCA：假设数据主要沿少数几个方向变化

它们不是同一种算法的不同版本，而是在回答完全不同的问题。
理解这一点，比背公式更重要。

## 训练时到底发生了什么

            KMeans 通过“分配样本 -> 更新中心”反复迭代；
DBSCAN 通过“局部密度是否足够高”扩展簇；
PCA 则是直接从数据协方差结构中找最重要的投影方向。

这意味着：

- KMeans 很依赖距离定义和特征缩放
- DBSCAN 很依赖 `eps` 和 `min_samples`
- PCA 更像线性代数问题，而不是传统意义上的迭代分类器训练

## 什么时候该用它

            使用建议：

- 想做用户分群、样本分群：优先考虑聚类
- 想做高维可视化、压缩特征：优先考虑 PCA
- 想找异常点、非球形簇：优先考虑 DBSCAN

## 最常见的误区

            常见误区：

1. **误以为聚类一定能找到“真实类别”**
   聚类找到的是“结构上的相似性”，不一定和人工标签完全一致。

2. **误以为降维后信息不会损失**
   降维本质就是压缩，只能尽量保留主信息，不可能无损。

3. **误以为 PCA 是聚类算法**
   PCA 是降维工具，不负责分组。

## 1. KMeans 的目标函数

$$
J = \sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i - \mu_k\|^2
$$

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]



print("临时目录:", temp_root)

In [ ]:
from contextlib import nullcontext
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import load_digits, load_wine, make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import sklearn.cluster._kmeans as kmeans_backend

# 当前 Windows 受限环境中 threadpoolctl 会在 KMeans 探测 BLAS 线程库时出错。
# 这里把线程池限制逻辑替换成空上下文，保证 notebook 可稳定执行。
kmeans_backend.threadpool_info = lambda: []
kmeans_backend.threadpool_limits = lambda *args, **kwargs: nullcontext()

wine = load_wine(as_frame=True)
X_wine = wine.data
y_wine = wine.target

scaler = StandardScaler()
X_wine_scaled = scaler.fit_transform(X_wine)

ks = range(2, 9)
kmeans_records = []
for k in ks:
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X_wine_scaled)
    kmeans_records.append(
        {"k": k, "inertia": model.inertia_, "silhouette": silhouette_score(X_wine_scaled, labels)}
    )

kmeans_eval_df = pd.DataFrame(kmeans_records)
kmeans_eval_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(kmeans_eval_df["k"], kmeans_eval_df["inertia"], marker="o")
axes[0].set_title("Elbow Method")

axes[1].plot(kmeans_eval_df["k"], kmeans_eval_df["silhouette"], marker="o", color="darkgreen")
axes[1].set_title("Silhouette Score")

plt.tight_layout()
plt.show()

In [ ]:
pca_wine = PCA(n_components=2, random_state=42)
wine_2d = pca_wine.fit_transform(X_wine_scaled)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(X_wine_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(x=wine_2d[:, 0], y=wine_2d[:, 1], hue=y_wine, palette="Set2", s=90, ax=axes[0])
axes[0].set_title("Wine 真实类别（PCA 2D）")

sns.scatterplot(x=wine_2d[:, 0], y=wine_2d[:, 1], hue=cluster_labels, palette="Set1", s=90, ax=axes[1])
axes[1].set_title("KMeans 聚类结果（PCA 2D）")

plt.tight_layout()
plt.show()

In [ ]:
subset_idx = np.random.choice(len(X_wine_scaled), size=35, replace=False)
linkage_matrix = linkage(X_wine_scaled[subset_idx], method="ward")

plt.figure(figsize=(16, 6))
dendrogram(linkage_matrix, leaf_rotation=90, leaf_font_size=10)
plt.title("Wine 子集的层次聚类树状图")
plt.show()

In [ ]:
X_moon, y_moon = make_moons(n_samples=500, noise=0.08, random_state=42)
X_moon_scaled = StandardScaler().fit_transform(X_moon)

dbscan = DBSCAN(eps=0.22, min_samples=5)
db_labels = dbscan.fit_predict(X_moon_scaled)

plt.figure(figsize=(10, 7))
sns.scatterplot(x=X_moon_scaled[:, 0], y=X_moon_scaled[:, 1], hue=db_labels, palette="tab10", s=70)
plt.title("DBSCAN 在双月数据上的聚类结果")
plt.show()

In [ ]:
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

X_digits_scaled = StandardScaler().fit_transform(X_digits)
pca_digits = PCA(n_components=20, random_state=42)
X_digits_pca = pca_digits.fit_transform(X_digits_scaled)

cumulative_ratio = np.cumsum(pca_digits.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].plot(range(1, 21), pca_digits.explained_variance_ratio_, marker="o")
axes[0].set_title("每个主成分的解释方差比")

axes[1].plot(range(1, 21), cumulative_ratio, marker="o", color="darkred")
axes[1].set_title("累计解释方差比")

plt.tight_layout()
plt.show()

pca_2d = PCA(n_components=2, random_state=42)
digits_2d = pca_2d.fit_transform(X_digits_scaled)

plt.figure(figsize=(10, 8))
sns.scatterplot(x=digits_2d[:, 0], y=digits_2d[:, 1], hue=y_digits, palette="tab10", s=55, alpha=0.8)
plt.title("Digits 数据的 PCA 二维投影")
plt.show()